# Cost-Aware LLM Routing — Phase 6 Full Batch Run (Kaggle)

Portable Kaggle version of `pipeline/batch_runner.py`. The module files below are
copy-pasted verbatim from the local repo (models/ollama_client.py,
eval/correctness.py, eval/llm_judge.py, pipeline/batch_runner.py) — same code,
no Kaggle-specific branching beyond the Ollama install/startup steps.

**Before running:**
1. In the notebook's Settings panel, enable **GPU** and **Internet**.
2. Upload your dataset file (`data/raw/queries_raw.jsonl`, or the curated version
   once available) as a Kaggle **Dataset** input to this notebook. Note the
   dataset slug shown under `/kaggle/input/<slug>/`.
3. Run all cells top to bottom.
4. When done, download `data/results/full_run.csv` and `full_run.jsonl` from the
   notebook's output/working directory and place them in your local repo's
   `data/results/` folder.


## 1. Install and start Ollama

**If this section fails with `FileNotFoundError: 'ollama'`**, it almost always
means the install step below couldn't reach the internet — go to
**Notebook → Settings → Internet** and switch it **On**, then re-run from the
top. The diagnostic cell right after the install command will tell you
explicitly whether the binary was actually installed before we try to use it.

The Ollama installer also needs `zstd` to extract its archive, which isn't in
Kaggle's base image — the cell below installs it first.


In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import shutil

ollama_path = shutil.which("ollama")
if ollama_path is None:
    raise RuntimeError(
        "ollama binary not found after the install step above. This almost always "
        "means Internet is disabled for this notebook. Go to Notebook -> Settings -> "
        "Internet and switch it On, then re-run all cells from the top."
    )
print(f"Found ollama at: {ollama_path}")
!ollama --version


In [ ]:
import subprocess
import time

import requests

ollama_proc = subprocess.Popen(
    [ollama_path, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

for _ in range(60):
    try:
        if requests.get("http://localhost:11434/api/tags", timeout=2).ok:
            print("Ollama server is up")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError(
        "Ollama server did not start in time. Check that the install step above "
        "succeeded (ollama_path should be a real path, not None)."
    )


## 2. Pull the three models (two routing tiers + judge)


In [ ]:
!ollama pull qwen2.5:0.5b-instruct-q4_0
!ollama pull qwen2.5:7b-instruct-q4_0
!ollama pull qwen2.5:14b


## 3. Recreate the project's Python modules (identical to the local repo)


In [ ]:
import os

for d in ["models", "eval", "pipeline", "data/raw", "data/results"]:
    os.makedirs(d, exist_ok=True)

for pkg in ["models", "eval", "pipeline"]:
    open(f"{pkg}/__init__.py", "w").close()


In [ ]:
%%writefile models/ollama_client.py
"""Thin wrapper around the local Ollama HTTP API for the two routing tiers,
plus a separate, more capable judge model used only for correctness grading
(never as a routing candidate) — see eval/llm_judge.py."""

from __future__ import annotations

import time
from dataclasses import dataclass

import requests

OLLAMA_BASE_URL = "http://localhost:11434"

TIER_MODELS = {
    # Originally qwen2.5:1.5b-instruct-q4_0; the pilot run showed it was
    # surprisingly capable (near-degenerate small/large split, see
    # results/writeup.md), so Tier 1 was moved down to 0.5b to restore a
    # genuine capability gap. Staying within the Qwen2.5 family keeps
    # instruction-tuning style constant and isolates parameter count as the
    # variable driving the capability difference.
    "small": "qwen2.5:0.5b-instruct-q4_0",
    "large": "qwen2.5:7b-instruct-q4_0",
}

# Deliberately distinct from both routing tiers: grading answers with the same
# model that produced one of them (Tier 2, in the original design) risks the
# judge being lenient toward its own reasoning style. qwen2.5:14b is only
# ever used for scoring, never as a routing candidate.
JUDGE_MODEL = "qwen2.5:14b"


@dataclass
class GenerationResult:
    tier: str | None
    model: str
    prompt: str
    response_text: str
    latency_seconds: float
    prompt_tokens: int
    completion_tokens: int


def _call_ollama(
    prompt: str,
    model: str,
    *,
    temperature: float = 0.0,
    timeout: float = 300.0,
    base_url: str = OLLAMA_BASE_URL,
    num_predict: int | None = None,
) -> tuple[dict, float]:
    options = {"temperature": temperature}
    if num_predict is not None:
        options["num_predict"] = num_predict
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": options,
    }
    start = time.perf_counter()
    resp = requests.post(f"{base_url}/api/generate", json=payload, timeout=timeout)
    resp.raise_for_status()
    latency = time.perf_counter() - start
    return resp.json(), latency


def generate(
    prompt: str,
    tier: str,
    *,
    temperature: float = 0.0,
    timeout: float = 300.0,
    base_url: str = OLLAMA_BASE_URL,
) -> GenerationResult:
    """Call the given routing tier's model ('small' or 'large').

    Uses temperature=0 by default for reproducible eval runs.
    """
    model = TIER_MODELS[tier]
    data, latency = _call_ollama(prompt, model, temperature=temperature, timeout=timeout, base_url=base_url)
    return GenerationResult(
        tier=tier,
        model=model,
        prompt=prompt,
        response_text=data.get("response", ""),
        latency_seconds=latency,
        prompt_tokens=data.get("prompt_eval_count", 0),
        completion_tokens=data.get("eval_count", 0),
    )


def generate_with_model(
    prompt: str,
    model: str,
    *,
    temperature: float = 0.0,
    timeout: float = 300.0,
    base_url: str = OLLAMA_BASE_URL,
    num_predict: int | None = None,
) -> GenerationResult:
    """Call an arbitrary Ollama model by name (used for the judge model)."""
    data, latency = _call_ollama(
        prompt, model, temperature=temperature, timeout=timeout, base_url=base_url, num_predict=num_predict
    )
    return GenerationResult(
        tier=None,
        model=model,
        prompt=prompt,
        response_text=data.get("response", ""),
        latency_seconds=latency,
        prompt_tokens=data.get("prompt_eval_count", 0),
        completion_tokens=data.get("eval_count", 0),
    )


def is_server_running(base_url: str = OLLAMA_BASE_URL) -> bool:
    try:
        resp = requests.get(f"{base_url}/api/tags", timeout=5)
        return resp.ok
    except requests.RequestException:
        return False


def ensure_models_pulled(base_url: str = OLLAMA_BASE_URL, include_judge: bool = True) -> None:
    """Raise if a required model isn't present in `ollama list`."""
    resp = requests.get(f"{base_url}/api/tags", timeout=10)
    resp.raise_for_status()
    available = {m["name"] for m in resp.json().get("models", [])}
    required = list(TIER_MODELS.values()) + ([JUDGE_MODEL] if include_judge else [])
    missing = [m for m in required if m not in available]
    if missing:
        raise RuntimeError(
            f"Missing Ollama models: {missing}. Run `ollama pull <model>` first."
        )


if __name__ == "__main__":
    ensure_models_pulled()
    for tier in ("small", "large"):
        result = generate("What is the capital of France?", tier=tier)
        print(f"[{tier}] {result.model} ({result.latency_seconds:.2f}s): {result.response_text.strip()}")


In [ ]:
%%writefile eval/correctness.py
"""Exact-match/rubric correctness scoring for objective, short-answer queries.

Only handles reference answers with a short "core" fact (a number, name, or
short phrase before any explanatory clause) — e.g. "Au", "1945", "150 miles
(60 x 2.5)" -> core "150 miles". Longer, open-ended reference answers (most
medium/hard queries) are not gradable this way; try_exact_match returns None
for those so the caller can fall back to eval/llm_judge.py.
"""

import re

_DELIM_RE = re.compile(r"[.,;(]|\bsince\b|\bbecause\b|\bwhich\b")
_NUM_RE = re.compile(r"-?\d+\.?\d*")


def _core_answer(reference_answer: str) -> str:
    match = _DELIM_RE.search(reference_answer)
    core = reference_answer[: match.start()] if match else reference_answer
    return core.strip()


def _normalize(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def try_exact_match(model_answer: str, reference_answer: str) -> bool | None:
    """Return True/False if confidently gradable by short-answer matching, else None."""
    # Guard on the *full* reference length, not just the extracted core: a
    # long multi-sentence procedural reference (e.g. a puzzle solution) can
    # have a short, coincidentally-numeric fragment before its first comma,
    # which would otherwise let a trivial number match (e.g. "3" from
    # "divide into groups of 3") falsely credit an unrelated or wrong answer.
    if len(_normalize(reference_answer).split()) > 20:
        return None

    core_norm = _normalize(_core_answer(reference_answer))
    words = core_norm.split()
    if not words or len(words) > 8:
        return None

    model_norm = _normalize(model_answer)

    # Try a literal phrase match first, but don't rely on it alone: model
    # answers often reformat numbers/units (e.g. LaTeX "\text{cm}^2" vs plain
    # "cm^2"), breaking a contiguous phrase match even when the underlying
    # answer is correct. Numeric answers get a fallback check regardless of
    # how many words the extracted core phrase has.
    pattern = r"\b" + re.escape(core_norm) + r"\b"
    if re.search(pattern, model_norm):
        return True

    core_nums = _NUM_RE.findall(core_norm)
    if core_nums and core_nums[0] in _NUM_RE.findall(model_norm):
        return True

    return False


In [ ]:
%%writefile eval/llm_judge.py
"""LLM-as-judge correctness scoring for open-ended queries.

Uses a dedicated judge model (JUDGE_MODEL in models/ollama_client.py) that is
distinct from both routing tiers, so grading isn't done by the same model
that produced one of the candidate answers. Manual validation of judge
verdicts against human judgment (30-40 examples, Section 9) still happens
separately in eval/validate_judge.py — do not skip that step before fully
trusting these labels.

judge_pair() grades both the small- and large-tier candidate answers in a
single call (instead of two separate judge() calls) to roughly halve judge
latency for queries where neither answer is exact-match gradable.
"""

import re

from models.ollama_client import JUDGE_MODEL, generate_with_model

_VERIFICATION_INSTRUCTIONS = """Check each candidate answer step by step:
1. Identify each factual claim, calculation, or step in the candidate answer.
2. Verify each one is actually correct and internally consistent (e.g. if it says a
   quantity is divided into groups, the group sizes must actually sum to the total;
   if it does arithmetic, the arithmetic must be right).
3. Confirm the candidate's final conclusion matches the reference answer's key facts
   and conclusion. Minor differences in wording or extra detail are fine as long as
   the substance and every material step are correct — but any factual, logical, or
   arithmetic error, or a self-contradictory step, makes the answer INCORRECT, even
   if the surface-level approach resembles the reference."""

JUDGE_PROMPT_TEMPLATE = """You are carefully grading a candidate answer to a question, against a reference answer.

Question: {query}

Reference answer: {reference}

Candidate answer: {candidate}

{instructions}

Respond with exactly one word on the first line: CORRECT or INCORRECT. Follow with a
one-sentence justification on a second line, citing the specific step that is right
or wrong.
""".replace("{instructions}", _VERIFICATION_INSTRUCTIONS)

JUDGE_PAIR_PROMPT_TEMPLATE = """You are carefully grading two independent candidate answers to a question, against a reference answer.

Question: {query}

Reference answer: {reference}

Candidate A: {candidate_a}

Candidate B: {candidate_b}

{instructions}

Grade Candidate A and Candidate B independently of each other. Respond in exactly
two lines, in this format:
A: CORRECT or INCORRECT
B: CORRECT or INCORRECT
""".replace("{instructions}", _VERIFICATION_INSTRUCTIONS)

_JUDGE_NUM_PREDICT = 150


def _parse_verdict(text: str) -> bool:
    first_line = text.splitlines()[0].strip().upper() if text else ""
    if "INCORRECT" in first_line:
        return False
    if "CORRECT" in first_line:
        return True
    return False  # unparseable judge output; treat conservatively as incorrect


def _parse_pair_verdict(text: str, label: str) -> bool:
    match = re.search(rf"^\s*{label}\s*[:\-]\s*(CORRECT|INCORRECT)", text, re.IGNORECASE | re.MULTILINE)
    if not match:
        return False  # unparseable; treat conservatively as incorrect
    return match.group(1).upper() == "CORRECT"


def judge(query_text: str, reference_answer: str, candidate_answer: str) -> tuple[bool, str]:
    prompt = JUDGE_PROMPT_TEMPLATE.format(query=query_text, reference=reference_answer, candidate=candidate_answer)
    result = generate_with_model(prompt, JUDGE_MODEL, temperature=0.0, num_predict=_JUDGE_NUM_PREDICT)
    text = result.response_text.strip()
    return _parse_verdict(text), text


def judge_pair(
    query_text: str, reference_answer: str, candidate_a: str, candidate_b: str
) -> tuple[bool, bool, str]:
    """Grade two candidate answers in a single judge call. Returns (verdict_a, verdict_b, raw_text)."""
    prompt = JUDGE_PAIR_PROMPT_TEMPLATE.format(
        query=query_text, reference=reference_answer, candidate_a=candidate_a, candidate_b=candidate_b
    )
    result = generate_with_model(prompt, JUDGE_MODEL, temperature=0.0, num_predict=_JUDGE_NUM_PREDICT)
    text = result.response_text.strip()
    return _parse_pair_verdict(text, "A"), _parse_pair_verdict(text, "B"), text


In [ ]:
%%writefile pipeline/batch_runner.py
"""Portable batch inference runner — identical code for local or Kaggle execution.

Runs each query through both Ollama tiers, logs latency and token counts,
scores correctness (exact-match where gradable, else LLM-as-judge), and
derives the empirical routing_label per Section 6:
  small correct & large correct   -> "small"
  small incorrect & large correct -> "large"
  small correct & large incorrect -> "small"
  both incorrect                  -> None (excluded from router training)

On Kaggle: install Ollama, start it as a background process, and pull both
models before invoking this script — no other environment-specific changes
are needed.
"""

import argparse
import random
from pathlib import Path

import pandas as pd

from eval.correctness import try_exact_match
from eval.llm_judge import judge, judge_pair
from models.ollama_client import ensure_models_pulled, generate, is_server_running


def load_queries(path: str) -> pd.DataFrame:
    p = Path(path)
    return pd.read_json(p, lines=True) if p.suffix == ".jsonl" else pd.read_csv(p)


def select_pilot_sample(df: pd.DataFrame, n_per_difficulty: int = 10, seed: int = 42) -> pd.DataFrame:
    rng = random.Random(seed)
    parts = []
    for _, group in df.groupby("difficulty_label"):
        idx = list(group.index)
        rng.shuffle(idx)
        parts.append(group.loc[idx[:n_per_difficulty]])
    return pd.concat(parts).sort_values("query_id").reset_index(drop=True)


def score_pair(
    query_text: str, reference_answer: str, small_answer: str, large_answer: str
) -> tuple[bool, str, bool, str]:
    """Score both candidate answers, using a single combined judge call when both
    fall through exact-match (the common case), to roughly halve judge latency."""
    small_result = try_exact_match(small_answer, reference_answer)
    large_result = try_exact_match(large_answer, reference_answer)

    if small_result is None and large_result is None:
        small_result, large_result, _ = judge_pair(query_text, reference_answer, small_answer, large_answer)
        return small_result, "llm_judge", large_result, "llm_judge"

    small_method = "exact_match"
    if small_result is None:
        small_result, _ = judge(query_text, reference_answer, small_answer)
        small_method = "llm_judge"

    large_method = "exact_match"
    if large_result is None:
        large_result, _ = judge(query_text, reference_answer, large_answer)
        large_method = "llm_judge"

    return small_result, small_method, large_result, large_method


def derive_routing_label(small_correct: bool, large_correct: bool) -> str | None:
    if small_correct:
        return "small"
    if large_correct:
        return "large"
    return None


def run_batch(input_path: str, output_path: str, limit: int | None = None, pilot: bool = False) -> pd.DataFrame:
    if not is_server_running():
        raise RuntimeError("Ollama server is not running at http://localhost:11434")
    ensure_models_pulled()

    df = load_queries(input_path)
    if pilot:
        df = select_pilot_sample(df)
    elif limit:
        df = df.head(limit)

    records = []
    for i, row in df.iterrows():
        query_text = row["query_text"]
        reference_answer = row["reference_answer"]
        print(f"[{len(records)+1}/{len(df)}] {row['query_id']} ({row['difficulty_label']}): {query_text[:60]}")

        small = generate(query_text, tier="small")
        large = generate(query_text, tier="large")

        small_correct, small_method, large_correct, large_method = score_pair(
            query_text, reference_answer, small.response_text, large.response_text
        )
        routing_label = derive_routing_label(small_correct, large_correct)

        record = dict(row)
        record.update(
            {
                "small_model_answer": small.response_text,
                "large_model_answer": large.response_text,
                "small_model_latency_seconds": small.latency_seconds,
                "large_model_latency_seconds": large.latency_seconds,
                "small_model_prompt_tokens": small.prompt_tokens,
                "small_model_completion_tokens": small.completion_tokens,
                "large_model_prompt_tokens": large.prompt_tokens,
                "large_model_completion_tokens": large.completion_tokens,
                "small_model_correct": small_correct,
                "large_model_correct": large_correct,
                "small_scoring_method": small_method,
                "large_scoring_method": large_method,
                "routing_label": routing_label,
            }
        )
        records.append(record)
        print(
            f"    small={'OK' if small_correct else 'X'} ({small_method}, {small.latency_seconds:.1f}s)  "
            f"large={'OK' if large_correct else 'X'} ({large_method}, {large.latency_seconds:.1f}s)  "
            f"routing_label={routing_label}"
        )

    out_df = pd.DataFrame(records)
    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_path, index=False)
    out_df.to_json(out_path.with_suffix(".jsonl"), orient="records", lines=True)
    print(f"\nWrote {len(out_df)} results to {out_path}")
    return out_df


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", default="data/raw/queries_raw.jsonl")
    parser.add_argument("--output", default="data/results/pilot_run.csv")
    parser.add_argument("--pilot", action="store_true", help="Run the 30-query stratified pilot sample")
    parser.add_argument("--limit", type=int, default=None)
    args = parser.parse_args()
    run_batch(args.input, args.output, limit=args.limit, pilot=args.pilot)


## 4. Locate the uploaded dataset

Looks for a `.jsonl` file under `/kaggle/input/` (the dataset you uploaded per
the instructions above). Falls back to `data/raw/queries_raw.jsonl` if you
copy/paste the dataset content there instead.


In [ ]:
import glob

candidates = sorted(glob.glob("/kaggle/input/**/*.jsonl", recursive=True))
INPUT_PATH = candidates[0] if candidates else "data/raw/queries_raw.jsonl"
print("Using dataset:", INPUT_PATH)


## 5. Run the full batch (identical code path to the local pilot run)


In [ ]:
import subprocess

result = subprocess.run(
    ["python", "-m", "pipeline.batch_runner", "--input", INPUT_PATH, "--output", "data/results/full_run.csv"],
    check=True,
)


## 6. Download results

`data/results/full_run.csv` and `data/results/full_run.jsonl` are now in the
notebook's working directory — download both from the Kaggle output panel and
copy them into your local repo's `data/results/` folder to continue with
Phase 7 (baselines) and Phase 8 (threshold sweep).


In [ ]:
import pandas as pd

df = pd.read_csv("data/results/full_run.csv")
print(f"Total records: {len(df)}")
print(df["routing_label"].value_counts(dropna=False))
